# 02. Bộ dữ liệu chuẩn và cách chia tập (dùng chung cho mọi mô hình)

Dựng **một** bộ dữ liệu chuẩn từ bảng ngày của notebook 01 (`src/data/`): điền thiếu nhân quả → đặc trưng → biến đổi log →
mẫu cửa sổ → vai trò train/val/test của từng fold. Chuẩn hoá z-score và ngưỡng Q90/Q95 được học **từ train của từng fold** khi
mô hình lấy dữ liệu (`UnifiedDataset.view`). Mọi mô hình — Persistence, Ridge-ARX, Extra Trees, XGBoost, RNN, LSTM, GRU — đọc
cùng bộ dữ liệu này, cùng danh sách mẫu, cùng ngày đích.

| Thành phần | Nội dung |
|---|---|
| `X_seq` | N × L × F (RNN / LSTM / GRU) |
| `X_flat` | N × (L·F) (Ridge-ARX / Extra Trees / XGBoost); Persistence dùng Q CĐ tại ngày gốc |
| `Y`, `M` | N × H đích log(1+Q) chuẩn hoá; mặt nạ ô đích có quan trắc thật |
| ngày gốc, vai trò | `results/dataset/samples_H1.csv`, `samples_H10.csv` |
| metadata | `results/dataset/features_meta.csv`, `dataset_meta.json` |

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src.data.windowing import UnifiedDataset, load_daily
from src.data.feature_engineering import metadata_table, feature_list
from src.data.splitting import fold_names
from src import pipeline
OUT = results_dir(cfg, 'dataset')
daily = load_daily(cfg)

## 1. Điền thiếu nhân quả và chọn ngưỡng khoảng thiếu G

| Trường hợp | Xử lý |
|---|---|
| Khoảng thiếu ≤ G ngày | Forward fill (giá trị quan trắc gần nhất trước đó) + cờ thiếu `*_miss` + tuổi số liệu `*_age` |
| Khoảng thiếu > G ngày | Không điền làm đầu vào: **loại mẫu** có cửa sổ 30 ngày chạm vào khoảng này |
| Q CĐ thiếu ở ngày đích | Không tạo nhãn giả; ô đích bị loại khỏi hàm mất mát và chấm điểm |

G ∈ {1, 3, 7} được chọn **một lần cho mọi mô hình** bằng CV 3 fold với LSTM và GRU (L = 14, hidden 64, 1 lớp, tập biến F1, bài
toán t+1). Để so sánh công bằng, mọi G được chấm trên **cùng** các mẫu val hợp lệ với cả ba ngưỡng (giao).

In [ ]:
gap_tab, G, gmean = pipeline.select_gap(cfg)
display(gap_tab.round(3)); gap_tab.to_csv(OUT / 'chon_nguong_khoang_thieu.csv', index=False, encoding='utf-8-sig')
print('NSE CV trung bình (LSTM, GRU) theo G:', gmean.round(4).to_dict(), '→ chọn G =', G)
# Đã có bộ dữ liệu đóng băng → giữ nguyên G của nó (manifest), không chọn lại
from src.data.freeze import frozen_dir
mf = frozen_dir(cfg) / 'manifest.json'
if mf.exists():
    G_dong_bang = json.loads(mf.read_text(encoding='utf-8'))['max_gap']
    if G != G_dong_bang:
        print(f'⚠ Lần chạy này chọn G = {G}, khác G = {G_dong_bang} của bộ dữ liệu đã đóng băng {cfg["dataset_version"]}.\n'
              '  NSE CV giữa các G chênh rất ít, và RNN/LSTM/GRU cho kết quả hơi khác trên GPU/CPU khác, nên lựa chọn có thể đổi.\n'
              f'  Giữ nguyên bộ dữ liệu đã đóng băng (G = {G_dong_bang}). Muốn dùng G mới thì tạo phiên bản dữ liệu mới (dataset_version).')
    G = G_dong_bang
print('G dùng cho bộ dữ liệu:', G)

## 2. Đặc trưng và các tập biến

In [ ]:
ds = {H: UnifiedDataset(cfg, H, G, daily) for H in (cfg['horizons']['main'], cfg['horizons']['extended'])}
d1 = ds[cfg['horizons']['main']]
meta = metadata_table(); display(meta); meta.to_csv(OUT / 'features_meta.csv', encoding='utf-8-sig')
fs = pd.DataFrame([{'tập biến': k, 'số biến': len(feature_list(cfg, k)), 'biến': ', '.join(feature_list(cfg, k))} for k in cfg['feature_sets']]).set_index('tập biến')
display(fs)

## 3. Mẫu và chia tập

| Tách | Train (ngày đích ≤) | Đánh giá |
|---|---|---|
| F1 | 31/12/2020 | val 2021 |
| F2 | 31/12/2021 | val 2022 (đỉnh 1.436 m³/s) |
| F3 | 31/12/2023 | val 2024 (đỉnh 2.802 m³/s) |
| FINAL | 31/12/2024 | test 01/2025 – 08/2026 (**khoá**: không dùng để chọn siêu tham số, kiến trúc, tập biến hay trọng số) |

Đầu vào của mẫu val/test được phép dùng quan trắc thuộc giai đoạn trước (như khi dự báo thực tế); chỉ ngày **đích** của train và
val/test được tách rời. Năm 2023 không làm fold val vì Cửa Đạt thiếu trọn mùa lũ (02/08 – 16/10/2023).

In [ ]:
for H, d in ds.items():
    print(f'— H = {H}: {len(d.S)} ngày gốc ({d.S.date.iloc[0].date()} → {d.S.date.iloc[-1].date()}); '
          f'loại do chạm khoảng thiếu > {G} ngày: {(~d.S.reliable).sum()}')
    display(d.summary().round(1))
    d.sample_table().to_csv(OUT / f'samples_H{H}.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(12, 2.6))
for i, sp in enumerate(fold_names(cfg) + ['FINAL']):
    r = d1.S[sp]
    for role, c in [('train', 'C0'), ('val', 'C1'), ('test', 'C3')]:
        x = d1.S.date[r == role]; ax.scatter(x, np.full(len(x), i), s=2, color=c, label=role if i == 0 or (role == 'test' and sp == 'FINAL') else None)
ax.set_yticks(range(4)); ax.set_yticklabels(fold_names(cfg) + ['FINAL']); ax.legend(markerscale=5, fontsize=8, loc='lower left')
ax.set_title('Vai trò của từng ngày gốc (t+1)'); plt.tight_layout(); plt.savefig(OUT / 'fig_chia_tap.png'); plt.show()

## 4. Kiểm tra thiết kế (tự động)

In [ ]:
checks = []
def check(name, cond):
    assert cond, name; checks.append(name)
f = d1.filled; obs = daily
for c in ['q_cd', 'hn_qxa', 'hn_qden', 'hn_mn']:
    check(f'điền nhân quả {c} = quan trắc gần nhất trước đó', np.allclose(obs[c].ffill(), f[c]))
check('tuổi số liệu chỉ dùng quá khứ', all(f.cd_age.iloc[t] == (0 if not obs.cd_missing.iloc[t] else f.cd_age.iloc[t - 1] + 1) for t in range(1, len(f))))
from src.data.quality_control import repeat_length_causal
x = obs.hn_qden.where(~obs.hn_missing).values
check('cờ lặp Q đến HN chỉ dùng dữ liệu ≤ t', np.array_equal(repeat_length_causal(x[:1000]), repeat_length_causal(x)[:1000]))
check('tổng mưa trượt chỉ dùng quá khứ', np.isclose(d1.F.mua_v1_7.iloc[500], obs.mua_v1.iloc[494:501].sum()))
for H, d in ds.items():
    tg = lambda ix: np.unique((d.S.date.values[ix][:, None] + np.arange(1, H + 1) * np.timedelta64(1, 'D')).ravel())
    for sp in fold_names(cfg) + ['FINAL']:
        v = d.view(sp, feature_list(cfg, 'F3'), 30)
        tr, ev = v.idx_train, v.idx_eval
        check(f'[H{H}/{sp}] ngày đích train và đánh giá không trùng', np.intersect1d(tg(tr), tg(ev)).size == 0)
        end = cfg['splits']['final']['train_end'] if sp == 'FINAL' else next(x for x in cfg['splits']['folds'] if x['name'] == sp)['train_end']
        check(f'[H{H}/{sp}] ngày đích train ≤ hết giai đoạn train', (d.S.date.values[tr] + np.timedelta64(H, 'D') <= np.datetime64(end)).all())
        td = d.train_days(sp); fz = feature_list(cfg, 'F3')
        mu, sd = d.T.loc[td, fz].mean(), d.T.loc[td, fz].std()
        bi = [c for c in fz if meta.loc[c, 'biến đổi'] == 'binary']; mu[bi], sd[bi] = 0.0, 1.0
        t_ = d.S.pos.iloc[ev[0]]
        check(f'[H{H}/{sp}] chuẩn hoá dùng μ, σ của ngày train (không chạm ngày đánh giá)',
              np.allclose(v.X_seq[ev[0], -1], ((d.T[fz].iloc[t_] - mu) / sd).values, atol=1e-4) and td[d.index > end].sum() == 0)
        check(f'[H{H}/{sp}] Q95 lấy từ train', np.isclose(v.q95, np.quantile(daily.q_cd[(daily.index <= end) & ~daily.cd_missing], .95)))
        check(f'[H{H}/{sp}] mẫu train/đánh giá không chạm khoảng thiếu dài', d.S.reliable.values[tr].all() and d.S.reliable.values[ev].all())
    i = 700; t = d.S.pos.iloc[i]; v = d.view('FINAL', feature_list(cfg, 'F3'), 14)
    sc_mu, sc_sd = d.T.loc[d.train_days('FINAL')].mean(), d.T.loc[d.train_days('FINAL')].std()
    check(f'[H{H}] bước cuối của cửa sổ là ngày gốc t', np.allclose(v.X_seq[i, -1, 0], (d.T.q_cd.iloc[t] - sc_mu.q_cd) / sc_sd.q_cd, atol=1e-5))
    check(f'[H{H}] đích = Q quan trắc t+1 … t+H', np.allclose(np.nan_to_num(v.QO[i]), np.nan_to_num(daily.q_cd.values[t + 1:t + H + 1])))
    check(f'[H{H}] test bắt đầu 01/01/2025 và không có ô đích thiếu', d.S.date[v.idx_eval].min() == pd.Timestamp('2025-01-01') and v.M[v.idx_eval].all())
check('mọi mô hình dùng cùng tập mẫu: X_flat là X_seq trải phẳng', np.allclose(v.X_flat[5], v.X_seq[5].ravel()))
for c in dict.fromkeys(checks): print('✔', c)
print(len(checks), 'phép kiểm tra đều đạt.')
(OUT / 'kiem_tra_thiet_ke.txt').write_text('\n'.join('✔ ' + c for c in dict.fromkeys(checks)) + f'\n{len(checks)} phép kiểm tra đều đạt.\n', encoding='utf-8')

## 5. Từ điển dữ liệu, đóng băng phiên bản và kiểm tra toàn vẹn

Bộ dữ liệu được **đóng băng** ở `data/frozen/v1/` (bảng ngày, đặc trưng, mẫu H1/H10, từ điển dữ liệu, báo cáo chia tập, cờ chất
lượng, `manifest.json` với SHA-256 từng file, hash cấu hình và hash mã `src/data`). Mọi notebook mô hình gọi
`pipeline.get_dataset()`: kiểm tra manifest và kiểm tra tập mẫu dựng lại trùng bản đóng băng trước khi huấn luyện. Bản đã đóng
băng không bị ghi đè — thay đổi dữ liệu phải tạo phiên bản mới (`dataset_version`).

In [ ]:
from src.data.dictionary import raw_dictionary, feature_dictionary
from src.data.freeze import freeze, verify
display(raw_dictionary())
saw = pd.read_csv(results_dir(cfg, 'phan_tich_du_lieu') / 'rang_cua_q_cd.csv')
suspect = pd.to_datetime(saw.loc[saw['đánh giá'].str.startswith('nghi'), 'ngày'])
man, created = freeze(cfg, G, suspect)
print('Tạo mới' if created else 'Đã có và trùng khớp —', 'phiên bản', man['version'], '| G =', man['max_gap'])
print(json.dumps({k: v for k, v in man.items() if k != 'files'}, ensure_ascii=False, indent=1))
display(pd.DataFrame(man['files']).T)
verify(cfg); print('✔ manifest khớp')

In [ ]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests/'], capture_output=True, text=True, cwd=ROOT)
(OUT / 'kiem_tra_toan_ven_pytest.txt').write_text(r.stdout + r.stderr, encoding='utf-8')
print(r.stdout[-1500:]); assert r.returncode == 0, 'unit test thất bại'

## 6. Đỉnh lũ trong tập test và lưu metadata

In [ ]:
peaks, thr = pipeline.test_peaks(cfg, d1)
print(f'Ngưỡng đỉnh = Q99 train cuối = {thr:.0f} m³/s; các đỉnh test:', [(p.date(), daily.q_cd[p]) for p in peaks])
meta_js = {'max_gap': G, 'gap_selection_nse_cv': {int(k): float(x) for k, x in gmean.items()},
           'n_samples': {H: len(d.S) for H, d in ds.items()}, 'test_peaks': [str(p.date()) for p in peaks], 'peak_threshold': thr}
(OUT / 'dataset_meta.json').write_text(json.dumps(meta_js, ensure_ascii=False, indent=1), encoding='utf-8')
print(meta_js)